# 重新加载数据

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import math

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.data_processing import load_marketing_data

df = load_marketing_data(PROJECT_ROOT / "data" / "raw" / "marketing_AB.csv")

df.head()

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14


# 汇总两组转化情况

In [2]:
ab_summary = df.groupby("test_group").agg(
    users=("user_id", "count"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean")
).reset_index()

ab_summary["conversion_rate_pct"] = ab_summary["conversion_rate"] * 100

ab_summary

,test_group,users,conversions,conversion_rate,conversion_rate_pct
0,ad,564577,14423,0.025547,2.554656
1,psa,23524,420,0.017854,1.785411


## 检验两组用户转化率是否存在统计显著差异

In [3]:
ad = ab_summary[ab_summary["test_group"] == "ad"].iloc[0]
psa = ab_summary[ab_summary["test_group"] == "psa"].iloc[0]

x_ad = ad["conversions"]
n_ad = ad["users"]

x_psa = psa["conversions"]
n_psa = psa["users"]

p_ad = x_ad / n_ad
p_psa = x_psa / n_psa

p_pool = (x_ad + x_psa) / (n_ad + n_psa)

standard_error = math.sqrt(
    p_pool * (1 - p_pool) * (1 / n_ad + 1 / n_psa)
)

z_score = (p_ad - p_psa) / standard_error

p_value = 2 * (1 - 0.5 * (1 + math.erf(abs(z_score) / math.sqrt(2))))

result = pd.DataFrame({
    "metric": [
        "ad_conversion_rate",
        "psa_conversion_rate",
        "absolute_lift",
        "relative_lift",
        "z_score",
        "p_value"
    ],
    "value": [
        p_ad,
        p_psa,
        p_ad - p_psa,
        (p_ad - p_psa) / p_psa,
        z_score,
        p_value
    ]
})

result

,metric,value
0,ad_conversion_rate,2.554656e-02
1,psa_conversion_rate,1.785411e-02
2,absolute_lift,7.692453e-03
3,relative_lift,4.308506e-01
4,z_score,7.370078e+00
5,p_value,1.705303e-13


## 95% 置信区间

In [4]:
se_unpooled = math.sqrt(
    p_ad * (1 - p_ad) / n_ad + 
    p_psa * (1 - p_psa) / n_psa
)

ci_low = (p_ad - p_psa) - 1.96 * se_unpooled
ci_high = (p_ad - p_psa) + 1.96 * se_unpooled

ci_result = pd.DataFrame({
    "metric": [
        "absolute_lift",
        "ci_low_95",
        "ci_high_95"
    ],
    "value": [
        p_ad - p_psa,
        ci_low,
        ci_high
    ]
})

ci_result

,metric,value
0,absolute_lift,0.007692
1,ci_low_95,0.005951
2,ci_high_95,0.009434


## A/B 测试结果

广告组的转化率高于 PSA 对照组。
广告组转化率约为 2.55%，PSA 对照组为 1.79%。
本研究检验两组转化率差异是否具备统计显著性，结果表明两组存在统计学意义上的差异，说明广告干预与更高的转化概率存在关联。
绝对提升幅度约 0.77 个百分点，相对提升幅度约 43%。
但由于广告曝光频次与转化行为高度相关，该结果需要结合后续的曝光与时间维度分析共同解读。

### 在曝光次数、星期、小时都被控制住以后，ad 组是否仍然比 psa 组更容易转化？

In [6]:
import statsmodels.formula.api as smf
import numpy as np

model_df = df.copy()

model_df["converted_int"] = model_df["converted"].astype(int)
model_df["is_ad"] = (model_df["test_group"] == "ad").astype(int)

model_df["log_total_ads"] = np.log1p(model_df["total_ads"])

logit_model = smf.logit(
    formula="converted_int ~ is_ad + log_total_ads + C(most_ads_day) + C(most_ads_hour)",
    data=model_df
).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.094961
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:          converted_int   No. Observations:               588101
Model:                          Logit   Df Residuals:                   588069
Method:                           MLE   Df Model:                           31
Date:                Sat, 19 Sep 2026   Pseudo R-squ.:                  0.1938
Time:                        17:08:49   Log-Likelihood:                -55846.
converged:                       True   LL-Null:                       -69267.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                       -8.5601      0.122    -70.093      0.000

# 提取广告效果

In [7]:
coef = logit_model.params["is_ad"]
p_value = logit_model.pvalues["is_ad"]

odds_ratio = np.exp(coef)

logit_result = pd.DataFrame({
    "metric": ["ad_coefficient", "odds_ratio", "p_value"],
    "value": [coef, odds_ratio, p_value]
})

logit_result

,metric,value
0,ad_coefficient,3.926148e-01
1,odds_ratio,1.480848e+00
2,p_value,3.748344e-14


### 假设所有用户的曝光次数、星期、小时分布都一样，只改变他们是否属于 ad 组，那么平均预测转化率会差多少？

In [8]:
ad_case = model_df.copy()
psa_case = model_df.copy()

ad_case["is_ad"] = 1
psa_case["is_ad"] = 0

adjusted_ad_rate = logit_model.predict(ad_case).mean()
adjusted_psa_rate = logit_model.predict(psa_case).mean()

adjusted_absolute_lift = adjusted_ad_rate - adjusted_psa_rate
adjusted_relative_lift = adjusted_absolute_lift / adjusted_psa_rate

adjusted_result = pd.DataFrame({
    "metric": [
        "adjusted_ad_conversion_rate",
        "adjusted_psa_conversion_rate",
        "adjusted_absolute_lift",
        "adjusted_relative_lift"
    ],
    "value": [
        adjusted_ad_rate,
        adjusted_psa_rate,
        adjusted_absolute_lift,
        adjusted_relative_lift
    ]
})

adjusted_result

,metric,value
0,adjusted_ad_conversion_rate,0.025542
1,adjusted_psa_conversion_rate,0.017927
2,adjusted_absolute_lift,0.007615
3,adjusted_relative_lift,0.424778


## 调整后的实验处理效应

通过逻辑回归对广告曝光频次、用户广告活跃高峰日、广告活跃高峰小时进行控制之后，广告组的预测转化率依旧高于 PSA 对照组。

调整后广告组转化率约为 2.55%，PSA 对照组约为 1.79%。
调整后的绝对提升约 0.76 个百分点，对应的调整后相对提升约 42.48%。

该结果与原始 A/B 测试得到的提升幅度十分接近，说明在纳入曝光与时间因素进行校正后，广告带来的正向实验效应依旧稳健。